# P1-E16.7A{renamed to E11.A  — Four-Endpoint Nested Conventional Selection and Frozen Refit

## Purpose

This notebook repairs two limitations identified during pre-submission review:

1. it adds the clinically relevant **Normal-versus-MCI** endpoint; and
2. it replaces representation/classifier selection on the small official validation partition with **nested development evaluation**.

Four endpoints are evaluated consistently: Normal versus AD, MCI versus AD, Normal versus MCI, and three-class Normal--MCI--AD classification. All preprocessing, representation selection, classifier selection, and hyperparameter selection occur inside the applicable training data. The official CAUEEG test partition is opened only after one configuration per endpoint has been refit and serialized.

Because the official test partition has been examined in earlier project iterations, its results are described as a **retrospective locked test evaluation**, not as prospectively untouched confirmation. No target-performance threshold is encoded in this notebook.

## 1. Configuration and environment

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, random, sys, time, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, matthews_corrcoef, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 20261005
OUTER_FOLDS = 5
INNER_FOLDS = 4
BOOTSTRAP_REPS = 2000
N_JOBS = -1

PROJECT_ROOT = Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUTS_ROOT = PROJECT_ROOT / 'paper1_outputs'
LEGACY_OUTPUTS_ROOT = Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
E07_ROOT = LEGACY_OUTPUTS_ROOT / 'p1_e07_1_full_cohort_reference_preprocessing_automated_qa'
E08_ROOT = LEGACY_OUTPUTS_ROOT / 'p1_e08_1_leakage_free_nonspiking_benchmark'
OUTPUT_ROOT = OUTPUTS_ROOT / 'p1_e16_7a_four_endpoint_nested_conventional_freeze'
TABLES = OUTPUT_ROOT / 'tables'
FIGURES = OUTPUT_ROOT / 'figures'
MODELS = OUTPUT_ROOT / 'models'
LOGS = OUTPUT_ROOT / 'logs'
for folder in (OUTPUT_ROOT, TABLES, FIGURES, MODELS, LOGS):
    folder.mkdir(parents=True, exist_ok=True)

# These are the exact authenticated inputs used by the original P1-E08.1
# conventional benchmark. Change them only if the same locked files were moved.
EXACT_FEATURE_NPZ = E08_ROOT / 'features' / '02_LOCKED_RECORDING_LEVEL_NONSPIKING_FEATURE_MATRIX.npz'
EXACT_PARTITION_MANIFEST = E07_ROOT / 'tables' / '09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'

ENDPOINTS = {
    'normal_vs_ad': ['Normal', 'AD'],
    'mci_vs_ad': ['MCI', 'AD'],
    'normal_vs_mci': ['Normal', 'MCI'],
    'three_class': ['Normal', 'MCI', 'AD'],
}
ENDPOINT_LABELS = {
    'normal_vs_ad': 'Normal vs AD',
    'mci_vs_ad': 'MCI vs AD',
    'normal_vs_mci': 'Normal vs MCI',
    'three_class': 'Three-class',
}
CHANCE_BA = {task: 1 / len(classes) for task, classes in ENDPOINTS.items()}

def seed_all(seed):
    random.seed(seed); np.random.seed(seed)
def sha256_file(path, block=2**20):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(block), b''): digest.update(chunk)
    return digest.hexdigest()
def save_table(frame, name):
    frame.to_csv(TABLES / f'{name}.csv', index=False)
    try: frame.to_excel(TABLES / f'{name}.xlsx', index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped: {exc}')
def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=600, bbox_inches='tight', facecolor='white')
    fig.savefig(FIGURES / f'{name}.pdf', bbox_inches='tight', facecolor='white')
def json_default(value):
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): return float(value)
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, Path): return str(value)
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')

seed_all(SEED)
sns.set_theme(style='whitegrid', context='paper', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 600, 'pdf.fonttype': 42,
                     'ps.fonttype': 42, 'axes.spines.top': False, 'axes.spines.right': False})
print('Output:', OUTPUT_ROOT)
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())

## 2. Resolve and authenticate the locked feature and partition inputs

In [ ]:
SEARCH_ROOTS = [OUTPUTS_ROOT, LEGACY_OUTPUTS_ROOT, PROJECT_ROOT]

def unique_by_content(paths, label):
    paths = sorted({Path(p) for p in paths if Path(p).is_file()})
    families = {}
    for path in paths:
        families.setdefault(sha256_file(path), []).append(path)
    if len(families) != 1:
        inventory = pd.DataFrame([
            {'path': str(path), 'sha256': sha256_file(path)} for path in paths
        ])
        display(inventory)
        raise RuntimeError(
            f'{label}: expected one content family, found {len(families)}. '
            'Set the corresponding EXACT_* path in the configuration cell.'
        )
    return sorted(next(iter(families.values())), key=lambda p: (len(str(p)), str(p)))[0]

def discover(exact, patterns, label, preferred_tokens=()):
    if exact is not None:
        path = Path(exact)
        if not path.is_file(): raise FileNotFoundError(path)
        return path
    found = []
    for root in SEARCH_ROOTS:
        if not root.is_dir(): continue
        for pattern in patterns:
            found.extend(root.rglob(pattern))
    preferred = [p for p in found if all(token.lower() in str(p).lower() for token in preferred_tokens)]
    return unique_by_content(preferred or found, label)

feature_path = discover(
    EXACT_FEATURE_NPZ,
    ['*LOCKED*FEATURE*MATRIX*.npz', '*feature*matrix*.npz', '*FEATURES*.npz'],
    'recording-level feature matrix', ('e08',)
)
partition_path = discover(
    EXACT_PARTITION_MANIFEST,
    ['09_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv',
     '*LOCKED*PREPROCESSING*ELIGIBLE*RECORDING*MANIFEST*.csv',
     '*PARTITION*MANIFEST*.csv', '*partition*manifest*.csv', '*COHORT*MANIFEST*.csv'],
    'official partition manifest', ('e07',)
)

inputs = pd.DataFrame([
    {'role': 'recording-level feature matrix', 'path': str(feature_path), 'sha256': sha256_file(feature_path)},
    {'role': 'official partition manifest', 'path': str(partition_path), 'sha256': sha256_file(partition_path)},
])
save_table(inputs, '01_INPUT_PROVENANCE_MANIFEST')
display(inputs)

## 3. Load and reconcile recording-level data

In [ ]:
def first_key(keys, candidates):
    lookup = {str(k).lower(): k for k in keys}
    for candidate in candidates:
        if candidate.lower() in lookup: return lookup[candidate.lower()]
    return None

with np.load(feature_path, allow_pickle=False) as archive:
    keys = list(archive.files)
    x_key = first_key(keys, ['X', 'features', 'feature_matrix'])
    id_key = first_key(keys, ['recording_id', 'recording_ids', 'series_id', 'ids'])
    name_key = first_key(keys, ['feature_names', 'features_names', 'columns'])
    group_key = first_key(keys, ['representation_group', 'representation_groups', 'feature_group'])
    label_key = first_key(keys, ['label', 'labels', 'diagnosis', 'experimental_label'])
    if not all([x_key, id_key, name_key]):
        raise KeyError(f'Feature NPZ keys {keys} do not contain X, recording identifiers and feature names.')
    X = np.asarray(archive[x_key], dtype=np.float32)
    recording_ids = np.asarray(archive[id_key]).astype(str)
    feature_names = np.asarray(archive[name_key]).astype(str)
    representation_groups = np.asarray(archive[group_key]).astype(str) if group_key else None
    embedded_labels = np.asarray(archive[label_key]).astype(str) if label_key else None

manifest = pd.read_csv(partition_path, dtype=str)
id_col = next((c for c in ['recording_id','series_id','record_id','id'] if c in manifest), None)
label_col = next((c for c in ['experimental_label','label','diagnosis','class'] if c in manifest), None)
split_col = next((c for c in ['split','partition','official_split','set'] if c in manifest), None)
if not id_col or not split_col:
    raise KeyError(f'Partition manifest columns are {manifest.columns.tolist()}; recording ID and split are required.')

def normalize_id(value):
    text = str(value).strip()
    return text.zfill(5) if text.replace('.', '', 1).isdigit() and '.' not in text else text
def normalize_label(value):
    text = str(value).strip().lower()
    aliases = {'normal':'Normal','control':'Normal','hc':'Normal','healthy':'Normal',
               'mci':'MCI','mild cognitive impairment':'MCI',
               'ad':'AD','alzheimers disease':'AD',"alzheimer's disease":'AD'}
    return aliases.get(text, str(value).strip())
def normalize_split(value):
    text = str(value).strip().lower()
    aliases = {'training':'train','tr':'train','validation':'validation','valid':'validation',
               'val':'validation','testing':'test','te':'test'}
    return aliases.get(text, text)

recording_ids = np.asarray([normalize_id(x) for x in recording_ids])
manifest['_recording_id'] = manifest[id_col].map(normalize_id)
manifest['_split'] = manifest[split_col].map(normalize_split)
if manifest['_recording_id'].duplicated().any():
    raise AssertionError('The partition manifest contains duplicate recording identifiers.')
lookup = manifest.set_index('_recording_id')
missing = sorted(set(recording_ids) - set(lookup.index))
if missing: raise AssertionError(f'{len(missing)} feature rows are absent from the partition manifest; examples: {missing[:10]}')

if label_col:
    labels = np.asarray([normalize_label(lookup.loc[rid, label_col]) for rid in recording_ids])
elif embedded_labels is not None:
    labels = np.asarray([normalize_label(x) for x in embedded_labels])
else:
    raise KeyError('No diagnostic label was available in the feature NPZ or partition manifest.')
splits = np.asarray([lookup.loc[rid, '_split'] for rid in recording_ids])

if X.ndim != 2 or X.shape != (len(recording_ids), len(feature_names)):
    raise AssertionError(f'Feature shape mismatch: X={X.shape}, IDs={len(recording_ids)}, names={len(feature_names)}')
if not np.isfinite(X).all():
    raise AssertionError('The locked feature matrix contains non-finite values; repair upstream, not here.')
if not set(np.unique(labels)).issubset({'Normal','MCI','AD'}):
    raise AssertionError(f'Unexpected labels: {sorted(set(labels))}')
if not set(np.unique(splits)).issubset({'train','validation','test'}):
    raise AssertionError(f'Unexpected partitions: {sorted(set(splits))}')

scope = (pd.DataFrame({'recording_id':recording_ids,'label':labels,'split':splits})
         .groupby(['split','label']).size().rename('n').reset_index())
save_table(scope, '02_FOUR_ENDPOINT_COHORT_SCOPE')
display(scope)
print('Matrix:', X.shape, '| Features:', len(feature_names))

## 4. Define feature representations and candidate models

Feature groups are determined from the authenticated feature names. Temporal features include Hjorth, line-length and related time-domain descriptors; spectral features include band-power and spectral descriptors; the hybrid representation contains all locked features. If the upstream schema uses different naming, update only the keyword lists and rerun before model fitting.

In [ ]:
lower_names = np.char.lower(feature_names.astype(str))
spectral_terms = ('bp_', 'bandpower', 'power_', 'psd', 'spectral', 'alpha', 'beta', 'theta', 'delta', 'gamma')
temporal_terms = ('hjorth', 'line_length', 'kurt', 'skew', 'entropy', 'rms', 'variance', 'std', 'complexity')
if representation_groups is not None and len(representation_groups) == len(feature_names):
    normalized_groups = np.char.lower(np.char.strip(representation_groups.astype(str)))
    spectral_idx = np.flatnonzero(np.char.find(normalized_groups, 'spectral') >= 0)
    temporal_idx = np.flatnonzero(np.char.find(normalized_groups, 'temporal') >= 0)
else:
    spectral_idx = np.flatnonzero([any(term in name for term in spectral_terms) for name in lower_names])
    temporal_idx = np.flatnonzero([any(term in name for term in temporal_terms) for name in lower_names])
if len(spectral_idx) == 0 or len(temporal_idx) == 0:
    raise AssertionError(
        f'Feature grouping failed (spectral={len(spectral_idx)}, temporal={len(temporal_idx)}). '
        'Inspect feature names and update the keyword lists without changing the locked matrix.'
    )
REPRESENTATIONS = {
    'spectral': spectral_idx,
    'temporal': temporal_idx,
    'hybrid': np.arange(len(feature_names)),
}
representation_table = pd.DataFrame([
    {'representation': name, 'n_features': len(indices)} for name, indices in REPRESENTATIONS.items()
])
save_table(representation_table, '03_REPRESENTATION_FEATURE_COUNTS')
display(representation_table)

def candidate_spaces(n_classes):
    logistic = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED)),
    ])
    svc = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('model', SVC(kernel='linear', class_weight='balanced', probability=True, random_state=SEED)),
    ])
    forest = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(n_estimators=400, class_weight='balanced_subsample',
                                         random_state=SEED, n_jobs=1)),
    ])
    return {
        'logistic_regression': (logistic, {'model__C': [0.01, 0.1, 1, 10]}),
        'linear_svc': (svc, {'model__C': [0.001, 0.01, 0.1, 1]}),
        'random_forest': (forest, {'model__max_features': ['sqrt', 0.3],
                                   'model__min_samples_leaf': [1, 3, 5]}),
    }

## 5. Nested development evaluation

The outer folds estimate the complete selection procedure. Within every outer-training partition, the inner folds select the representation, classifier family and hyperparameters. The outer-held-out recordings are never used in this selection. Because CAUEEG participant identity is unavailable, separation is guaranteed only at the recording-identifier level.

In [ ]:
def safe_n_splits(y, requested, context):
    minimum = int(pd.Series(y).value_counts().min())
    folds = min(requested, minimum)
    if folds < 2: raise ValueError(f'{context}: fewer than two observations in the smallest class.')
    return folds

def align_probability(estimator, X_eval, estimator_classes, required_classes):
    raw = np.asarray(estimator.predict_proba(X_eval), float)
    aligned = np.zeros((len(X_eval), len(required_classes)), float)
    mapping = {str(label): i for i, label in enumerate(estimator_classes)}
    for j, label in enumerate(required_classes):
        if label not in mapping: raise AssertionError(f'Missing probability class {label}.')
        aligned[:, j] = raw[:, mapping[label]]
    if not np.allclose(aligned.sum(axis=1), 1, atol=1e-6): raise AssertionError('Probabilities do not sum to one.')
    return aligned

selection_rows, prediction_rows, split_rows, candidate_rows = [], [], [], []
development_mask = np.isin(splits, ['train','validation'])
test_mask = splits == 'test'

for endpoint, classes in ENDPOINTS.items():
    endpoint_mask = np.isin(labels, classes)
    dev_indices = np.flatnonzero(development_mask & endpoint_mask)
    y_dev = labels[dev_indices]
    outer_k = safe_n_splits(y_dev, OUTER_FOLDS, f'{endpoint} outer CV')
    outer = StratifiedKFold(n_splits=outer_k, shuffle=True, random_state=SEED)
    print(f'\n{endpoint}: n={len(dev_indices)}, outer folds={outer_k}')
    for fold, (outer_train_local, outer_test_local) in enumerate(outer.split(dev_indices, y_dev), 1):
        train_idx = dev_indices[outer_train_local]
        holdout_idx = dev_indices[outer_test_local]
        y_train, y_holdout = labels[train_idx], labels[holdout_idx]
        inner_k = safe_n_splits(y_train, INNER_FOLDS, f'{endpoint} fold {fold} inner CV')
        inner = StratifiedKFold(n_splits=inner_k, shuffle=True, random_state=SEED + fold)
        fold_candidates = []
        for representation, columns in REPRESENTATIONS.items():
            for family, (estimator, grid) in candidate_spaces(len(classes)).items():
                search = GridSearchCV(estimator, grid, scoring='balanced_accuracy', cv=inner,
                                      n_jobs=N_JOBS, refit=True, return_train_score=False,
                                      error_score='raise')
                search.fit(X[train_idx][:, columns], y_train)
                row = {'endpoint': endpoint, 'outer_fold': fold, 'representation': representation,
                       'classifier': family, 'inner_balanced_accuracy': float(search.best_score_),
                       'best_params_json': json.dumps(search.best_params_, sort_keys=True),
                       'n_outer_train': len(train_idx), 'n_outer_holdout': len(holdout_idx)}
                fold_candidates.append((row, search.best_estimator_, columns))
                candidate_rows.append(row)
        # Deterministic tie rule: score, fewer features, representation name, classifier name.
        fold_candidates.sort(key=lambda item: (-round(item[0]['inner_balanced_accuracy'], 12),
                                               len(item[2]), item[0]['representation'], item[0]['classifier']))
        selected, fitted, columns = fold_candidates[0]
        probabilities = align_probability(fitted, X[holdout_idx][:, columns], fitted.classes_, classes)
        predicted = np.asarray(classes)[probabilities.argmax(axis=1)]
        outer_ba = balanced_accuracy_score(y_holdout, predicted)
        selection_rows.append({**selected, 'outer_balanced_accuracy': float(outer_ba),
                               'n_features': len(columns)})
        for row_index, true_label, pred_label, probability in zip(holdout_idx, y_holdout, predicted, probabilities):
            row = {'endpoint': endpoint, 'outer_fold': fold,
                   'recording_id': recording_ids[row_index], 'true_label': true_label,
                   'predicted_label': pred_label, 'selected_representation': selected['representation'],
                   'selected_classifier': selected['classifier']}
            row.update({f'probability_{label}': float(probability[j]) for j, label in enumerate(classes)})
            prediction_rows.append(row)
        for row_index in train_idx:
            split_rows.append({'endpoint': endpoint, 'outer_fold': fold,
                               'recording_id': recording_ids[row_index], 'role': 'outer_train'})
        for row_index in holdout_idx:
            split_rows.append({'endpoint': endpoint, 'outer_fold': fold,
                               'recording_id': recording_ids[row_index], 'role': 'outer_holdout'})
        print(' fold', fold, '|', selected['representation'], selected['classifier'],
              '| inner BA', round(selected['inner_balanced_accuracy'], 3),
              '| outer BA', round(outer_ba, 3))

candidate_results = pd.DataFrame(candidate_rows)
outer_selection = pd.DataFrame(selection_rows)
oof_predictions = pd.DataFrame(prediction_rows)
split_audit = pd.DataFrame(split_rows)
save_table(candidate_results, '04_INNER_CANDIDATE_RESULTS')
save_table(outer_selection, '05_OUTER_FOLD_SELECTION_AND_PERFORMANCE')
save_table(oof_predictions, '06_NESTED_OOF_PREDICTIONS')
save_table(split_audit, '07_OUTER_SPLIT_AUDIT')
display(outer_selection.round(4))

## 6. Nested performance estimates and recording-bootstrap intervals

In [ ]:
def metrics_from_predictions(frame, classes):
    y_true = frame.true_label.to_numpy()
    y_pred = frame.predicted_label.to_numpy()
    matrix = frame[[f'probability_{label}' for label in classes]].to_numpy(float)
    out = {
        'n': len(frame),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }
    try:
        if len(classes) == 2:
            out['roc_auc'] = roc_auc_score((y_true == classes[1]).astype(int), matrix[:, 1])
        else:
            encoded = pd.Categorical(y_true, categories=classes, ordered=True).codes
            out['roc_auc'] = roc_auc_score(encoded, matrix, labels=np.arange(len(classes)),
                                           multi_class='ovr', average='macro')
    except ValueError:
        out['roc_auc'] = np.nan
    return out

metric_rows, interval_rows = [], []
rng = np.random.default_rng(SEED)
for endpoint, classes in ENDPOINTS.items():
    frame = oof_predictions[oof_predictions.endpoint.eq(endpoint)].reset_index(drop=True)
    observed = metrics_from_predictions(frame, classes)
    metric_rows.append({'endpoint': endpoint, 'evaluation_role': 'nested_development_oof', **observed})
    strata = {label: np.flatnonzero(frame.true_label.to_numpy() == label) for label in classes}
    boot = []
    for _ in range(BOOTSTRAP_REPS):
        sampled = np.concatenate([rng.choice(indices, len(indices), replace=True) for indices in strata.values()])
        boot.append(metrics_from_predictions(frame.iloc[sampled], classes))
    boot = pd.DataFrame(boot)
    for metric, estimate in observed.items():
        if metric == 'n': continue
        values = boot[metric].dropna()
        interval_rows.append({'endpoint': endpoint, 'evaluation_role': 'nested_development_oof',
                              'metric': metric, 'estimate': estimate,
                              'ci95_low': values.quantile(.025), 'ci95_high': values.quantile(.975)})

nested_metrics = pd.DataFrame(metric_rows)
nested_intervals = pd.DataFrame(interval_rows)
save_table(nested_metrics, '08_NESTED_DEVELOPMENT_METRICS')
save_table(nested_intervals, '09_NESTED_BOOTSTRAP_INTERVALS')
display(nested_metrics.round(4))

## 7. Freeze one endpoint configuration using development data only

For each endpoint, the final representation/classifier family is selected from its mean inner-CV performance across outer folds. Hyperparameters are then selected by inner cross-validation on the complete train-plus-validation development cohort. The resulting fitted pipeline is serialized before the official test labels are evaluated.

In [ ]:
frozen_rows = []
for endpoint, classes in ENDPOINTS.items():
    summary = (candidate_results[candidate_results.endpoint.eq(endpoint)]
               .groupby(['representation','classifier'], as_index=False)
               .agg(mean_inner_ba=('inner_balanced_accuracy','mean'),
                    sd_inner_ba=('inner_balanced_accuracy','std')))
    summary['n_features'] = summary.representation.map(lambda name: len(REPRESENTATIONS[name]))
    summary = summary.sort_values(['mean_inner_ba','n_features','representation','classifier'],
                                  ascending=[False,True,True,True]).reset_index(drop=True)
    winner = summary.iloc[0]
    representation, family = winner.representation, winner.classifier
    columns = REPRESENTATIONS[representation]
    estimator, grid = candidate_spaces(len(classes))[family]
    endpoint_dev = np.flatnonzero(development_mask & np.isin(labels, classes))
    y_dev = labels[endpoint_dev]
    final_inner_k = safe_n_splits(y_dev, INNER_FOLDS, f'{endpoint} final inner CV')
    final_inner = StratifiedKFold(n_splits=final_inner_k, shuffle=True, random_state=SEED + 100)
    final_search = GridSearchCV(estimator, grid, scoring='balanced_accuracy', cv=final_inner,
                                n_jobs=N_JOBS, refit=True, error_score='raise')
    final_search.fit(X[endpoint_dev][:, columns], y_dev)
    bundle = {
        'endpoint': endpoint,
        'classes': classes,
        'representation': representation,
        'classifier': family,
        'selected_feature_indices': columns,
        'selected_feature_names': feature_names[columns],
        'full_feature_schema_sha256': hashlib.sha256('\n'.join(feature_names).encode()).hexdigest(),
        'best_params': final_search.best_params_,
        'inner_balanced_accuracy': float(final_search.best_score_),
        'development_recording_ids': recording_ids[endpoint_dev],
        'estimator': final_search.best_estimator_,
        'seed': SEED,
        'selection_protocol': 'nested development selection; final inner CV on train plus validation',
    }
    model_path = MODELS / f'{endpoint}_frozen_conventional.joblib'
    joblib.dump(bundle, model_path)
    frozen_rows.append({'endpoint': endpoint, 'representation': representation,
                        'classifier': family, 'n_features': len(columns),
                        'final_inner_balanced_accuracy': float(final_search.best_score_),
                        'best_params_json': json.dumps(final_search.best_params_, sort_keys=True),
                        'model_path': str(model_path), 'model_sha256': sha256_file(model_path)})

frozen_models = pd.DataFrame(frozen_rows)
save_table(frozen_models, '10_FROZEN_ENDPOINT_MODELS')
display(frozen_models)

## 8. Retrospective locked official-test evaluation

The serialized endpoint models are applied without refitting, feature substitution, threshold optimization, or recalibration. This test evaluation is kept separate from nested development OOF estimates. It is not described as untouched because the official partition was examined in earlier project iterations.

In [ ]:
test_prediction_rows = []
for frozen in frozen_models.itertuples(index=False):
    endpoint = frozen.endpoint
    classes = ENDPOINTS[endpoint]
    bundle = joblib.load(frozen.model_path)
    endpoint_test = np.flatnonzero(test_mask & np.isin(labels, classes))
    probabilities = align_probability(bundle['estimator'],
                                       X[endpoint_test][:, bundle['selected_feature_indices']],
                                       bundle['estimator'].classes_, classes)
    predicted = np.asarray(classes)[probabilities.argmax(axis=1)]
    for row_index, pred_label, probability in zip(endpoint_test, predicted, probabilities):
        row = {'endpoint': endpoint, 'recording_id': recording_ids[row_index],
               'true_label': labels[row_index], 'predicted_label': pred_label,
               'evaluation_role': 'retrospective_locked_official_test'}
        row.update({f'probability_{label}': float(probability[j]) for j, label in enumerate(classes)})
        test_prediction_rows.append(row)

test_predictions = pd.DataFrame(test_prediction_rows)
test_metric_rows, test_interval_rows = [], []
rng = np.random.default_rng(SEED + 1)
for endpoint, classes in ENDPOINTS.items():
    frame = test_predictions[test_predictions.endpoint.eq(endpoint)].reset_index(drop=True)
    observed = metrics_from_predictions(frame, classes)
    test_metric_rows.append({'endpoint': endpoint, 'evaluation_role': 'retrospective_locked_official_test', **observed})
    strata = {label: np.flatnonzero(frame.true_label.to_numpy() == label) for label in classes}
    boot = []
    for _ in range(BOOTSTRAP_REPS):
        sampled = np.concatenate([rng.choice(indices, len(indices), replace=True) for indices in strata.values()])
        boot.append(metrics_from_predictions(frame.iloc[sampled], classes))
    boot = pd.DataFrame(boot)
    for metric, estimate in observed.items():
        if metric == 'n': continue
        values = boot[metric].dropna()
        test_interval_rows.append({'endpoint': endpoint, 'evaluation_role': 'retrospective_locked_official_test',
                                   'metric': metric, 'estimate': estimate,
                                   'ci95_low': values.quantile(.025), 'ci95_high': values.quantile(.975)})

test_metrics = pd.DataFrame(test_metric_rows)
test_intervals = pd.DataFrame(test_interval_rows)
save_table(test_predictions, '11_RETROSPECTIVE_LOCKED_TEST_PREDICTIONS')
save_table(test_metrics, '12_RETROSPECTIVE_LOCKED_TEST_METRICS')
save_table(test_intervals, '13_RETROSPECTIVE_LOCKED_TEST_INTERVALS')
display(test_metrics.round(4))

## 9. Publication-ready comparisons

In [ ]:
combined = pd.concat([nested_metrics, test_metrics], ignore_index=True)
ba_intervals = pd.concat([nested_intervals, test_intervals], ignore_index=True)
ba_intervals = ba_intervals[ba_intervals.metric.eq('balanced_accuracy')].copy()
ba_intervals['endpoint_label'] = ba_intervals.endpoint.map(ENDPOINT_LABELS)
ba_intervals['design_label'] = ba_intervals.evaluation_role.map({
    'nested_development_oof': 'Nested development OOF',
    'retrospective_locked_official_test': 'Retrospective locked test',
})
save_table(combined, '14_EVIDENCE_STRATIFIED_FOUR_ENDPOINT_SUMMARY')

fig, axes = plt.subplots(1, 4, figsize=(16, 4.8), sharey=True)
for ax, endpoint in zip(axes, ENDPOINTS):
    frame = ba_intervals[ba_intervals.endpoint.eq(endpoint)].reset_index(drop=True)
    for i, row in frame.iterrows():
        error = [[row.estimate-row.ci95_low], [row.ci95_high-row.estimate]]
        ax.errorbar(row.estimate, i, xerr=error, fmt='o', capsize=4,
                    color='#0072B2' if i == 0 else '#D55E00')
        ax.text(min(row.ci95_high + .015, .96), i, f'{row.estimate:.3f}', va='center', fontsize=8)
    ax.axvline(CHANCE_BA[endpoint], color='0.35', ls='--', lw=1.2,
               label=f'Chance BA = {CHANCE_BA[endpoint]:.3f}')
    ax.set_yticks(range(len(frame)), frame.design_label if ax is axes[0] else [])
    ax.set_xlim(.25, 1.0); ax.set_title(ENDPOINT_LABELS[endpoint]); ax.set_xlabel('Balanced accuracy')
    ax.legend(frameon=False, fontsize=7, loc='lower right')
axes[0].set_ylabel('Evaluation design')
fig.suptitle('Four-endpoint nested development and retrospective locked-test performance', fontweight='bold')
save_figure(fig, 'Figure_01_Four_Endpoint_Nested_and_Locked_Test_Performance')
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(15, 3.9))
for ax, endpoint in zip(axes, ENDPOINTS):
    classes = ENDPOINTS[endpoint]
    frame = test_predictions[test_predictions.endpoint.eq(endpoint)]
    matrix = confusion_matrix(frame.true_label, frame.predicted_label, labels=classes)
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(ENDPOINT_LABELS[endpoint]); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
fig.suptitle('Retrospective locked official-test confusion matrices', fontweight='bold')
save_figure(fig, 'Figure_02_Four_Endpoint_Locked_Test_Confusion_Matrices')
plt.show()

## 10. Integrity and provenance lock

In [ ]:
def endpoint_probability_check(frame):
    details = []
    passed = True
    for endpoint, classes in ENDPOINTS.items():
        subset = frame.loc[frame.endpoint.eq(endpoint)].copy()
        columns = [f'probability_{label}' for label in classes]
        missing = [column for column in columns if column not in subset.columns]
        if subset.empty or missing:
            passed = False
            details.append({'endpoint': endpoint, 'n': len(subset), 'missing_columns': missing,
                            'finite': False, 'sum_to_one': False})
            continue
        probabilities = subset[columns].to_numpy(float)
        finite = bool(np.isfinite(probabilities).all())
        sum_to_one = bool(np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6))
        passed = passed and finite and sum_to_one
        details.append({'endpoint': endpoint, 'n': len(subset), 'missing_columns': [],
                        'finite': finite, 'sum_to_one': sum_to_one})
    return bool(passed), details

oof_probability_pass, oof_probability_detail = endpoint_probability_check(oof_predictions)
test_probability_pass, test_probability_detail = endpoint_probability_check(test_predictions)

checks = []
for endpoint, classes in ENDPOINTS.items():
    expected_dev = set(recording_ids[development_mask & np.isin(labels, classes)])
    observed_dev = oof_predictions.loc[oof_predictions.endpoint.eq(endpoint), 'recording_id']
    checks.extend([
        {'check': f'{endpoint}: each development recording predicted once',
         'pass': not observed_dev.duplicated().any() and set(observed_dev) == expected_dev,
         'detail': f'observed={len(observed_dev)}, expected={len(expected_dev)}'},
        {'check': f'{endpoint}: all required development classes present',
         'pass': set(oof_predictions.loc[oof_predictions.endpoint.eq(endpoint), 'true_label']) == set(classes),
         'detail': classes},
        {'check': f'{endpoint}: frozen model authenticated',
         'pass': sha256_file(Path(frozen_models.set_index('endpoint').loc[endpoint, 'model_path'])) ==
                 frozen_models.set_index('endpoint').loc[endpoint, 'model_sha256'],
         'detail': frozen_models.set_index('endpoint').loc[endpoint, 'model_sha256']},
    ])

overlap = (split_audit.groupby(['endpoint','outer_fold','recording_id']).role.nunique() > 1).any()
checks.extend([
    {'check': 'four endpoints completed', 'pass': set(nested_metrics.endpoint) == set(ENDPOINTS),
     'detail': sorted(nested_metrics.endpoint.tolist())},
    {'check': 'Normal-versus-MCI completed', 'pass': 'normal_vs_mci' in set(nested_metrics.endpoint),
     'detail': 'early-stage cross-sectional endpoint'},
    {'check': 'no outer train-holdout identity overlap', 'pass': not overlap,
     'detail': 'recording-identifier level'},
    {'check': 'binary chance references equal 0.500',
     'pass': all(np.isclose(CHANCE_BA[e], .5) for e in ENDPOINTS if len(ENDPOINTS[e]) == 2),
     'detail': CHANCE_BA},
    {'check': 'three-class chance reference equals one third',
     'pass': np.isclose(CHANCE_BA['three_class'], 1/3), 'detail': CHANCE_BA['three_class']},
    {'check': 'official test excluded from model selection',
     'pass': set(recording_ids[test_mask]).isdisjoint(set(split_audit.recording_id)),
     'detail': 'verified from split audit'},
    {'check': 'OOF endpoint-specific probabilities finite and sum to one',
     'pass': oof_probability_pass, 'detail': oof_probability_detail},
    {'check': 'test endpoint-specific probabilities finite and sum to one',
     'pass': test_probability_pass, 'detail': test_probability_detail},
])
integrity = pd.DataFrame(checks)
save_table(integrity, '15_FINAL_INTEGRITY_CHECKS')
display(integrity)
if not integrity['pass'].all():
    display(integrity.loc[~integrity['pass']])
    raise AssertionError('P1-E16.7A integrity failure.')

artifact_rows = []
for path in sorted(OUTPUT_ROOT.rglob('*')):
    if path.is_file() and path.name != '16_OUTPUT_MANIFEST.csv':
        artifact_rows.append({'relative_path': str(path.relative_to(OUTPUT_ROOT)),
                              'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
output_manifest = pd.DataFrame(artifact_rows)
save_table(output_manifest, '16_OUTPUT_MANIFEST')

run_record = {
    'experiment': 'P1-E16.7A',
    'status': 'P1-E16.7A PASSED — FOUR-ENDPOINT NESTED CONVENTIONAL SELECTION AND FREEZE',
    'completed_utc': datetime.now(timezone.utc).isoformat(),
    'endpoints': list(ENDPOINTS),
    'nested_outer_folds_requested': OUTER_FOLDS,
    'nested_inner_folds_requested': INNER_FOLDS,
    'seed': SEED,
    'official_test_used_for_selection': False,
    'official_test_claim': 'retrospective locked evaluation; not prospectively untouched',
    'input_manifest_sha256': sha256_file(TABLES / '01_INPUT_PROVENANCE_MANIFEST.csv'),
    'frozen_models': frozen_models.to_dict('records'),
    'integrity_passed': True,
    'probability_validation': 'endpoint-specific required columns; finite; row sums equal one',
    'next_step': 'P1-E16.7B frozen four-endpoint external evaluation and shift diagnostics',
}
(OUTPUT_ROOT / 'run_record.json').write_text(
    json.dumps(run_record, indent=2, default=json_default), encoding='utf-8')
print(run_record['status'])

## Takeaways

- Normal versus MCI is now evaluated consistently as a cross-sectional early-stage discrimination endpoint.
- The nested OOF estimates evaluate the complete conventional representation/classifier selection procedure.
- The official test is separated from development selection and truthfully labelled retrospective rather than untouched.
- One authenticated conventional model is frozen per endpoint for the next external and shift-diagnostic stage.
- These results do not establish prediction of MCI-to-AD conversion or clinical readiness.